<a href="https://colab.research.google.com/github/semillero-computacion-cuantica-urosario/Qiskit/blob/main/VQC_bandido_multibrazo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión práctica: Clasificadores Cuánticos Variacionales (VQC)
### Del concepto al problema del bandido multibrazo — Qiskit


In [ ]:
# En Colab:
%pip install -q qiskit scikit-learn matplotlib pylatexenc

Notebook diseñado para la versión 2.5.2 de Qiskit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
import qiskit
print("Qiskit", qiskit.__version__)

---
# Parte A — El VQC como concepto

Un VQC (Variational Quantum Classifier) es un modelo de aprendizaje supervisado en el que un circuito cuántico parametrizado actúa como función de hipótesis $f_\theta(\mathbf x)$ y un optimizador clásico ajusta sus parámetros. Formalmente es una familia de modelos

$$f_\theta(\mathbf x) = \langle \psi(\mathbf x,\theta)|\,M\,|\psi(\mathbf x,\theta)\rangle,\qquad |\psi(\mathbf x,\theta)\rangle = W(\theta)\,U_\Phi(\mathbf x)\,|0\rangle^{\otimes n},$$

es decir, el valor esperado de un observable sobre un estado que depende del dato y de los parámetros.

Las cuatro piezas clave son:

1. **Feature map** $U_\Phi(\mathbf{x})$: codifica el dato clásico en un estado cuántico (fijo, sin parámetros entrenables). Hay varias formas de hacerlo, en este caso usaremos un qubit por feature en el dataset.
2. **Ansatz** $W(\boldsymbol\theta)$: circuito parametrizado que se entrena.
3. **Medición + interpretación**: la regla de Born da probabilidades sobre bitstrings $b$. Para clasificación binaria usamos la paridad:
$$
p_\theta(y=1\mid\mathbf{x}) = \sum_{b:\ \mathrm{paridad}(b)=1} |\langle b|\psi(\mathbf{x},\boldsymbol\theta)\rangle|^2 = \frac{1-\langle Z\otimes Z\rangle}{2}.
$$
4. **Optimizador clásico**: minimiza la entropía cruzada
$$
\mathcal{L}(\boldsymbol\theta) = -\frac1N\sum_i \big[y_i\log p_\theta(1|\mathbf x_i) + (1-y_i)\log(1-p_\theta(1|\mathbf x_i))\big].
$$
La entropía cruzada mide qué tan sorprendido está el modelo con las etiquetas verdaderas. Minimizar $\mathcal L$ obliga al modelo a dar probabilidad alta a la clase correcta en todos los datos, y castiga con mucha fuerza las equivocaciones cometidas con confianza.

El computador cuántico solo evalúa probabilidades; el ciclo de optimización es clásico.

## A.1 Datos: *two moons*
Para el ejercicio usaremos el set de datos [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) de la librería [scikit-learn](https://scikit-learn.org/stable/index.html).

Escalamos las características a $[0,\pi]$ porque entrarán como ángulos.


In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X, y = make_moons(200, noise=0.1, random_state=0) # explorar X, y
X = MinMaxScaler((0, np.pi)).fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1)

plt.scatter(*Xtr.T, c=ytr, cmap='coolwarm', edgecolor='k', s=25)  # qué hace *
plt.xlabel('$x_0$'); plt.ylabel('$x_1$'); plt.title('Entrenamiento'); plt.show()

## A.2 Feature map tipo ZZ ([Havlíček et al., 2019](https://arxiv.org/pdf/1804.11326))

Cada repetición aplica $H^{\otimes n}$, luego $P(2x_i)$ en cada qubit y el término de interacción $e^{-i\phi_{01} Z_0Z_1}$ con $\phi_{01}=(\pi-x_0)(\pi-x_1)$, implementado como CX–$P(2\phi_{01})$–CX.

La compuerta $P$ es una compuerta de fase y tiene representación matricial

$$P(\lambda) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\lambda} \end{pmatrix}, \qquad P(2x_i) = \begin{pmatrix} 1 & 0 \\ 0 & e^{2ix_i} \end{pmatrix}.$$

Cabe observar que es una fase relativa y no global.

In [ ]:
def zz_feature_map(n, reps=2):
    x = ParameterVector('x', n)
    qc = QuantumCircuit(n, name='U_Φ')
    for _ in range(reps):
        qc.h(range(n))
        for i in range(n):
            qc.p(2 * x[i], i)
        for i in range(n - 1):
            qc.cx(i, i + 1) # (control, target)
            qc.p(2 * (np.pi - x[i]) * (np.pi - x[i + 1]), i + 1)
            qc.cx(i, i + 1)
    return qc, x

def ry_ansatz(n, reps=3, name='θ'):
    # Capas de RY + escalera de CX (equivalente a RealAmplitudes)
    th = ParameterVector(name, n * (reps + 1))
    qc = QuantumCircuit(n, name='W'); k = 0
    for r in range(reps + 1): # recorre las capas
        for i in range(n):
            qc.ry(th[k], i); k += 1
        if r < reps:
            for i in range(n - 1):
                qc.cx(i, i + 1)
    return qc, th


Ahora ejecutamos el feature map y el ansatz:

* **n:**	número de qubits.	2, uno por característica de make_moons.
* **fm:**	circuito del feature map $U_\Phi(\mathbf x)$.	QuantumCircuit de 2 qubits con 2 repeticiones de H, P y CX–P–CX.
* **xp:**	parámetros de los datos.	ParameterVector con x[0] y x[1].
* **an:**	circuito del ansatz $W(\theta)$.	QuantumCircuit de 2 qubits con 4 capas de $R_Y$ y CX entre ellas.
* **tp:**	parámetros entrenables	ParameterVector con θ[0] … θ[7], es decir $2\times(3+1)=8$.
* **vqc:**	circuito completo $W(\theta)\,U_\Phi(\mathbf x)$.	**fm** seguido de **an**.

In [ ]:
n = 2
fm, xp = zz_feature_map(n, reps=2)
an, tp = ry_ansatz(n, reps=3)
vqc = fm.compose(an)
vqc.draw('mpl')

## A.3 El modelo y la pérdida
Usamos simulación exacta (`Statevector`) para entrenar rápido. Después verificamos con shots.

In [ ]:
PARITY = np.array([bin(b).count('1') % 2 for b in range(2**n)])

def make_model(circ, xparams, tparams):
    def p1(x, th):
        bind = {**dict(zip(xparams, x)), **dict(zip(tparams, th))}
        return Statevector(circ.assign_parameters(bind)).probabilities() @ PARITY # producto punto
    return p1

def cross_entropy(p1, th, X, y):
    p = np.clip([p1(x, th) for x in X], 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# Entrenamiento del modelo. Busca los pesos θ que minimizan la entropía
def train(p1, n_params, maxiter=150, seed=0):
    th0 = np.random.default_rng(seed).uniform(-np.pi, np.pi, n_params)
    hist = []
    def f(th):
        L = cross_entropy(p1, th, Xtr, ytr); hist.append(L); return L
    res = minimize(f, th0, method='COBYLA', options={'maxiter': maxiter})
    return res.x, hist

def accuracy(p1, th, X, y):
    return np.mean((np.array([p1(x, th) for x in X]) > 0.5) == y)

In [ ]:
p1_zz = make_model(vqc, xp, tp) # Produce función de probabilidad
th_zz, hist_zz = train(p1_zz, len(tp), maxiter=150) # Pesos entrenados y lista de pérdidas
print(f"Acc. entrenamiento: {accuracy(p1_zz, th_zz, Xtr, ytr):.3f}")
print(f"Acc. prueba:        {accuracy(p1_zz, th_zz, Xte, yte):.3f}")
plt.plot(hist_zz); plt.xlabel('evaluación'); plt.ylabel('pérdida'); plt.show()

In [ ]:
np.min(hist_zz)

Evaluar el porcentaje de aciertos como
$$100\times e^{-\min(\text{hist_zz})} $$



In [ ]:
#
100*np.exp(-np.min(hist_zz))

In [ ]:
def plot_frontera(p1, th, titulo, res=40):
    g = np.linspace(0, np.pi, res) # cuadrícula
    Z = np.array([[p1([a, b], th) for a in g] for b in g]) # Evalúa el modelo en cada punto
    plt.contourf(g, g, Z, levels=20, cmap='coolwarm', alpha=0.6)
    plt.contour(g, g, Z, levels=[0.5], colors='k') # frontera de predicciones
    plt.scatter(*Xte.T, c=yte, cmap='coolwarm', edgecolor='k', s=25)
    plt.title(titulo); plt.xlabel('$x_0$'); plt.ylabel('$x_1$'); plt.show()

plot_frontera(p1_zz, th_zz, 'VQC con feature map ZZ')

In [ ]:
len(yte)

Pregunta para discutir. Aumentar reps del ansatz casi no mejora el resultado (≈ 80 %). ¿Por qué?

Cada dato entra al circuito una cantidad fija de veces, así que  pθ(1|x)  es un polinomio trigonométrico en  x  con un espectro de frecuencias determinado por el feature map, no por el ansatz (Schuld, Sweke y Meyer, 2021). El ansatz solo ajusta los coeficientes de Fourier.

## A.4 Data re-uploading
Intercalamos la codificación con las capas entrenables: $\big[W_L(\theta)\,S(\mathbf x)\big]\cdots\big[W_1(\theta)\,S(\mathbf x)\big]|00\rangle$. Más repeticiones de $S(\mathbf x)$ ⇒ más frecuencias accesibles.

In [ ]:
def reupload_circuit(n, L, final_layer=False):
    x = ParameterVector('x', n)
    th = ParameterVector('θ', 2 * n * (L + int(final_layer)))
    qc = QuantumCircuit(n); k = 0
    for l in range(L):
        for i in range(n):
            qc.ry(x[i], i)                       # codificación S(x)
        for i in range(n):
            qc.ry(th[k], i); qc.rz(th[k + 1], i); k += 2
        qc.cx(0, 1)
    if final_layer:                              # capa entrenable final (se usa en la Parte B)
        for i in range(n):
            qc.ry(th[k], i); qc.rz(th[k + 1], i); k += 2
    return qc, x, th

ru, xr, tr = reupload_circuit(2, L=3)
ru.draw('mpl')

In [ ]:
p1_ru = make_model(ru, xr, tr)
th_ru, hist_ru = train(p1_ru, len(tr), maxiter=300)
print(f"Acc. entrenamiento: {accuracy(p1_ru, th_ru, Xtr, ytr):.3f}")
print(f"Acc. prueba:        {accuracy(p1_ru, th_ru, Xte, yte):.3f}")
plot_frontera(p1_ru, th_ru, 'VQC con data re-uploading (L=3)')

¿Cambia el porcentaje de aciertos?

## A.5 Del simulador exacto a los *shots*
En hardware no tenemos $\langle Z\otimes Z\rangle$ exacto: lo estimamos con un número finito de mediciones. Usamos el primitivo `StatevectorSampler`.

In [ ]:
def p1_shots(circ, xparams, tparams, th, X, shots=1024):
    qcm = circ.copy(); qcm.measure_all()
    params = list(qcm.parameters)
    vals = []
    for x in X:
        bind = {**dict(zip(xparams, x)), **dict(zip(tparams, th))}
        vals.append([bind[p] for p in params])
    res = StatevectorSampler().run([(qcm, np.array(vals))], shots=shots).result()[0]
    out = []
    for i in range(len(X)):
        counts = res.data.meas.get_counts(i)
        out.append(sum(c for b, c in counts.items() if b.count('1') % 2) / shots)
    return np.array(out)

for shots in [16, 128, 1024]:
    acc = np.mean((p1_shots(ru, xr, tr, th_ru, Xte, shots) > 0.5) == yte)
    print(f"shots = {shots:5d}  ->  acc. prueba = {acc:.3f}")

### Ejercicios A
1. Cambie la interpretación de paridad por "probabilidad de que el qubit 0 sea 1". ¿Cambia la frontera?
2. Repita el entrenamiento de re-uploading con 3 semillas distintas. ¿Qué tan sensible es a la inicialización?
3. Entrene **usando shots** dentro de la función de pérdida (con COBYLA o SPSA). ¿Qué pasa con la convergencia?

---
# Parte B — El bandido multibrazo

En cada ronda $t=1,\dots,T$ se elige un brazo $a_t\in\{0,\dots,K-1\}$ y se recibe una recompensa $r_t\sim\mathrm{Bernoulli}(\mu_{a_t})$, con $\mu_k$ **incógnitas**. El objetivo es minimizar el *regret* acumulado:
$$
R_T = T\mu^* - \sum_{t=1}^T \mu_{a_t},\qquad \mu^*=\max_k\mu_k.
$$
El dilema central: **explorar** (probar brazos poco conocidos) vs **explotar** (jugar el que parece mejor).

**Diferencia clave con clasificación:** en un VQC supervisado conocemos la etiqueta correcta. En el bandido solo observamos la recompensa **del brazo que jugamos** (*bandit feedback*).

## B.1 Referencias clásicas: ε-greedy y UCB1

In [ ]:
MU = np.array([0.2, 0.45, 0.7, 0.5]); K = len(MU); T = 800

def eps_greedy(seed, eps=0.1):
    rng = np.random.default_rng(seed); n = np.zeros(K); q = np.zeros(K); mu_t = []
    for t in range(T):
        a = rng.integers(K) if rng.random() < eps else np.argmax(q)
        r = rng.random() < MU[a]; n[a] += 1; q[a] += (r - q[a]) / n[a]; mu_t.append(MU[a])
    return np.cumsum(MU.max() - np.array(mu_t))

def ucb1(seed):
    rng = np.random.default_rng(seed); n = np.zeros(K); q = np.zeros(K); mu_t = []
    for t in range(T):
        a = t if t < K else np.argmax(q + np.sqrt(2 * np.log(t) / n))
        r = rng.random() < MU[a]; n[a] += 1; q[a] += (r - q[a]) / n[a]; mu_t.append(MU[a])
    return np.cumsum(MU.max() - np.array(mu_t))

reg_eps = np.array([eps_greedy(s) for s in range(20)])
reg_ucb = np.array([ucb1(s) for s in range(20)])
print("Regret final  ε-greedy:", reg_eps[:, -1].mean().round(1), "  UCB1:", reg_ucb[:, -1].mean().round(1))

## B.2 Una política cuántica (sin contexto)

Con $n=2$ qubits tenemos $2^n=4$ resultados posibles ⇒ 4 brazos. Definimos la política
$$
\pi_\theta(a) = |\langle a|W(\theta)|00\rangle|^2 .
$$
- La regla de Born da **gratis** una distribución de probabilidad válida.
- **Elegir un brazo = hacer una sola medición** (1 shot).
- Inicializamos en la distribución uniforme: $R_Y(\pi/2)$ en ambos qubits en la primera capa, el resto en 0.

⚠️ Qiskit usa orden *little-endian*: el brazo $a$ corresponde al entero del bitstring $q_1q_0$.

**Entrenamiento: REINFORCE (gradiente de política)**
$$
\nabla_\theta J = \mathbb{E}_{a\sim\pi_\theta}\big[(r-b)\,\nabla_\theta\log\pi_\theta(a)\big],\qquad \nabla\log\pi = \frac{\nabla\pi}{\pi},
$$
donde $b$ es una línea base (promedio móvil de recompensas) que reduce la varianza. Como cada $\theta_j$ aparece en una única rotación de Pauli, usamos la **regla de corrimiento de parámetros**:
$$
\frac{\partial \pi_\theta(a)}{\partial\theta_j} = \frac{\pi_{\theta+\frac{\pi}{2}e_j}(a) - \pi_{\theta-\frac{\pi}{2}e_j}(a)}{2}.
$$

In [ ]:
pol, tpol = ry_ansatz(2, reps=2, name='φ')
display(pol.draw('mpl'))

def policy_probs(th):
    return Statevector(pol.assign_parameters(dict(zip(tpol, th)))).probabilities()

def grad_pi(prob_fn, th, a):
    # Regla de corrimiento de parámetros para ∂π(a)/∂θ_j
    g = np.zeros(len(th))
    for j in range(len(th)):
        e = np.zeros(len(th)); e[j] = np.pi / 2
        g[j] = (prob_fn(th + e)[a] - prob_fn(th - e)[a]) / 2
    return g

th_unif = np.zeros(len(tpol)); th_unif[:2] = np.pi / 2
print("π inicial:", policy_probs(th_unif).round(3))

In [ ]:
def quantum_bandit(seed, lr=0.05):
    rng = np.random.default_rng(seed)
    th = th_unif.copy(); base = 0.0; mu_t = []; hist_pi = []
    for t in range(T):
        pi = policy_probs(th)
        a = rng.choice(K, p=pi)                    # "medir" = elegir brazo
        r = float(rng.random() < MU[a])
        th += lr * (r - base) * grad_pi(policy_probs, th, a) / max(pi[a], 1e-6)
        base += 0.05 * (r - base)
        mu_t.append(MU[a]); hist_pi.append(pi)
    return np.cumsum(MU.max() - np.array(mu_t)), np.array(hist_pi)

runs = [quantum_bandit(s) for s in range(5)]
reg_q = np.array([r[0] for r in runs])
print("Regret final política cuántica:", reg_q[:, -1].mean().round(1))
print("Brazo final más probable por semilla:", [int(np.argmax(r[1][-1])) for r in runs])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for reg, lab in [(reg_eps, 'ε-greedy'), (reg_ucb, 'UCB1'), (reg_q, 'Política cuántica')]:
    m, s = reg.mean(0), reg.std(0)
    ax[0].plot(m, label=lab); ax[0].fill_between(range(T), m - s, m + s, alpha=0.2)
ax[0].set_xlabel('t'); ax[0].set_ylabel('regret acumulado'); ax[0].legend()
for k in range(K):
    ax[1].plot(runs[0][1][:, k], label=f'brazo {k} (μ={MU[k]})')
ax[1].set_xlabel('t'); ax[1].set_ylabel('π_θ(a)'); ax[1].legend(); ax[1].set_title('Evolución de la política (semilla 0)')
plt.tight_layout(); plt.show()

**Para discutir.**
- Repita con `lr=0.15`. En algunas semillas la política converge al brazo con μ = 0.5: el gradiente de política puede **colapsar la exploración prematuramente**. UCB1 tiene garantías ($R_T = O(\log T)$); REINFORCE no.
- ¿Qué tiene de "cuántico" esta política? En el fondo es una parametrización de una distribución sobre 4 elementos. Lo interesante aparece cuando la política depende de un **contexto** codificado en el circuito.

## B.3 Bandido contextual: el VQC como política

Ahora en cada ronda llega un contexto $\mathbf{x}_t\in[0,\pi]^2$ y la política es exactamente un VQC con 4 salidas:
$$
\pi_\theta(a\mid\mathbf{x}) = |\langle a|\,W(\theta)\,U(\mathbf{x})\,|00\rangle|^2 .
$$

**Entorno.** Sean $b_0=[x_0>\pi/2]$ y $b_1=[x_1>\pi/2]$. El mejor brazo es
$$
a^*(\mathbf x) = b_0 + 2\,(b_0\oplus b_1),
$$
que recompensa con probabilidad 0.8; los demás brazos, con 0.2. El XOR obliga a que el circuito **entrelace** la información de ambas coordenadas.

| | Clasificación (Parte A) | Bandido contextual |
|---|---|---|
| Entrada | $\mathbf x$ | $\mathbf x$ |
| Salida | clase | brazo (acción) |
| Señal de entrenamiento | etiqueta correcta | solo la recompensa del brazo jugado |
| Pérdida | entropía cruzada | $-\mathbb E[r]$ vía REINFORCE |

Usamos el circuito de re-uploading con $L=1$ y una capa entrenable final (8 parámetros), y para reducir la varianza promediamos el gradiente en minilotes de 10 rondas y actualizamos con Adam.

In [ ]:
ctx_circ, xc, tc = reupload_circuit(2, L=1, final_layer=True)
display(ctx_circ.draw('mpl'))

def best_arm(x):
    b0, b1 = int(x[0] > np.pi / 2), int(x[1] > np.pi / 2)
    return b0 + 2 * (b0 ^ b1)

def reward_prob(x, a):
    return 0.8 if a == best_arm(x) else 0.2

def ctx_probs(x, th):
    bind = {**dict(zip(xc, x)), **dict(zip(tc, th))}
    return Statevector(ctx_circ.assign_parameters(bind)).probabilities()

In [ ]:
def contextual_bandit(seed=1, T=1500, B=10, lr=0.1):
    rng = np.random.default_rng(seed)
    th = rng.uniform(-0.1, 0.1, len(tc)); base = 0.35
    m = np.zeros(len(th)); v = np.zeros(len(th)); k = 0        # Adam
    mu_t = []
    for it in range(T // B):
        G = np.zeros(len(th))
        for _ in range(B):
            x = rng.uniform(0, np.pi, 2)
            pi = ctx_probs(x, th)
            a = rng.choice(4, p=pi)
            r = float(rng.random() < reward_prob(x, a))
            G += (r - base) * grad_pi(lambda t_: ctx_probs(x, t_), th, a) / max(pi[a], 1e-6) / B
            base += 0.02 * (r - base)
            mu_t.append(reward_prob(x, a))
        k += 1; m = 0.9 * m + 0.1 * G; v = 0.999 * v + 0.001 * G**2
        th += lr * (m / (1 - 0.9**k)) / (np.sqrt(v / (1 - 0.999**k)) + 1e-8)   # ascenso
    return th, np.array(mu_t)

th_ctx, mu_ctx = contextual_bandit()

In [ ]:
Xeval = np.random.default_rng(99).uniform(0, np.pi, (400, 2))
acc_greedy = np.mean([np.argmax(ctx_probs(x, th_ctx)) == best_arm(x) for x in Xeval])
print(f"Precisión de la política greedy: {acc_greedy:.3f}")

w = 100
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(np.convolve(mu_ctx, np.ones(w) / w, mode='valid'), label='VQC-política')
ax[0].axhline(0.8, ls='--', c='g', label='oráculo'); ax[0].axhline(0.35, ls='--', c='r', label='aleatoria')
ax[0].set_xlabel('t'); ax[0].set_ylabel('recompensa esperada (media móvil)'); ax[0].legend()

g = np.linspace(0, np.pi, 40)
A_true = np.array([[best_arm([a, b]) for a in g] for b in g])
A_pol = np.array([[np.argmax(ctx_probs([a, b], th_ctx)) for a in g] for b in g])
for axi, A, tit in [(ax[1], A_true, 'Mejor brazo (verdad)'), (ax[2], A_pol, 'Brazo greedy de la política')]:
    im = axi.imshow(A, origin='lower', extent=[0, np.pi, 0, np.pi], cmap='tab10', vmin=0, vmax=9)
    axi.set_title(tit); axi.set_xlabel('$x_0$'); axi.set_ylabel('$x_1$')
plt.tight_layout(); plt.show()

**Para discutir.**
- La recompensa media de la política estocástica queda por debajo del oráculo, aunque la política greedy acierte casi siempre: el agente **sigue explorando**. ¿Cómo bajar la temperatura de la política?
- Pruebe varias semillas: la precisión greedy oscila aproximadamente entre 0.75 y 0.97. ¿Es un problema de la arquitectura, del optimizador o de la varianza de REINFORCE?

---
# Parte C — Discusión y ejercicios

**¿Hay ventaja cuántica aquí?** No. Las políticas de esta sesión son parametrizaciones (interesantes) de distribuciones de probabilidad, y un modelo clásico pequeño haría lo mismo. Los resultados con ventaja demostrable en bandidos usan **acceso tipo oráculo** a los brazos y **estimación de amplitud**, que da mejoras cuadráticas en identificación del mejor brazo y regret logarítmico en ciertos modelos.

**Lecturas sugeridas**
- Havlíček et al., *Supervised learning with quantum-enhanced feature spaces*, Nature (2019).
- Pérez-Salinas et al., *Data re-uploading for a universal quantum classifier*, Quantum (2020).
- Schuld, Sweke y Meyer, *Effect of data encoding on the expressive power of variational quantum machine-learning models*, PRA (2021).
- Casalé et al., *Quantum bandits*, Quantum Machine Intelligence (2020).
- Wang, You, Li y Childs, *Quantum exploration algorithms for multi-armed bandits*, AAAI (2021).

### Ejercicios B
1. **¿Importa el entrelazamiento?** Elimine el `cx` de `reupload_circuit` y vuelva a entrenar el bandido contextual. Compare la precisión greedy. Luego cambie el entorno a $a^*=b_0+2b_1$ (sin XOR) y repita.
2. **Exploración.** Añada una regularización de entropía $\beta\,H(\pi_\theta(\cdot|\mathbf x))$ al objetivo y estudie su efecto en la Parte B.2 con `lr=0.15`.
3. **Referencia clásica contextual.** Implemente ε-greedy por cuadrante (tabla 4×4) y compare. ¿Qué información "sabe" ese agente que el VQC no?
4. **Shots.** Reemplace `Statevector` por estimaciones con shots en el gradiente. ¿Cuántos shots por evaluación hacen falta para que el aprendizaje siga funcionando?